# PROC SUMMARY による通信収益保証サマリーキューブの構築

## エグゼクティブサマリー

無線キャリアの収益保証チームは、1か月分の加入者・日次単位の請求レコードをコンパクトなサマリーキューブに事前集計し、アナリストが生のファクトテーブルを再スキャンすることなく、地域・プラン階層・通話種別ごとに収益をドリルダウンできるようにする。`PROC SUMMARY` は100件の加入者・日次レコードを多次元セルの集合にロールアップする。最も細かい粒度(地域 x プラン階層 x 通話種別)は27通り中25個のセルを埋め、名前付きサブキューブがアナリストが最も頻繁に照会するマージナル(周辺集計)を提供する。このサンプル月では、キャリアは3つのアクティブな地域全体で\$222.78を決済しており、南部(\$97.44)と東部(\$86.94)を合わせると収益の83%を占め、北部は\$38.40にとどまる。最も収益の大きい単一サブキューブはプラス階層の音声サービス(18件で\$59.06)であり、セルをレコード当たり収益でランク付けすると、データ通信セルが漏えい監査の最有力候補として浮かび上がる(最高収益 \$7.87/件)。以下のすべての数値は実行結果から直接読み取ったものである。

## データソース

| データセット | 行数 | 粒度 | キー変数 |
|---------|------|-------|---------------|
| `work.cdr_billing` | 100 | 加入者・日次利用サマリー1行 | `region`(東部/南部/北部)、`plan_tier`(プリペイド/ベーシック/プラス)、`call_type`(音声/SMS/データ)、`device_class`、`bill_month`、`revenue`、`call_minutes`、`data_mb`、`subscriber_wt` |
| `work.cube_nway` | 25 | 空でない (region x plan_tier x call_type) セル1行 | `_FREQ_`、`rev_total`、`rev_mean`、`rev_max`、`min_total`、`data_total` |
| `work.cube_hier` | 22 | 名前付きドリルサブキューブのセル1行 | `_TYPE_`、`_FREQ_`、`rev_total` |

すべてのデータは `call streaminit()` / `rand()` によりインラインで生成される。外部ファイルやネットワークアクセスは不要である。この環境はライセンスなしで動作するため、書き込まれるデータセットは100観測に制限される — このシナリオは、その上限内でキューブが完全に埋まるようにサイズ設定されている。

## 生の通話明細レコードからドリル可能なキューブへ

無線キャリアは、日々数百万件の利用イベントにわたって収益を決済する。「先月の南部地域におけるプラス階層の音声収益はいくらだったか?」といった質問に、毎回生のファクトテーブルを再スキャンせずに答えられるようにするため、収益保証アナリストのためにデータを**事前集計**する。

`PROC SUMMARY` はまさにこの用途のための Base SAS の主力ツールである。フラットなファクトテーブルを1つ以上の `CLASS` 次元でグループ化し、要求された統計量を出力データセットに書き込み、各行に `_TYPE_`(どの次元がアクティブか)と `_FREQ_`(セルの背後にあるレコード数)のタグを付ける。この出力データセットこそが多次元キューブである — OLAP ツールが公開するのと同じロールアップ構造を、印刷・結合・スライスが可能な通常の SAS データセットとして具現化したものだ。

このノートブックでは:

1. 現実的な1か月分の加入者・日次請求レコードを生成する。
2. `PROC SUMMARY NWAY` で最も細かい粒度のキューブ(地域 x プラン階層 x 通話種別)を構築する。
3. `TYPES` 文で名前付きの**ドリルサブキューブ**を具現化する。
4. `WEIGHT` で収益を加入者ベースに投影し、1つの地域にドリルダウンし、漏えいトリアージのためにレコード当たり収益でセルをランク付けする。

## ステップ1 - 合成の通話明細/請求データを生成する

各行は、ある加入者がある日に利用した1つのサービスの利用状況を要約する。再現性のために `call streaminit` を使い、妥当な分布を描くために `rand()` を使う。収益と利用量はプラン階層に応じてスケールし、音声収益は課金対象分数を、データ収益はメガバイト数を追跡する。各 `RAND('table', ...)` にはカテゴリごとに1つの確率が与えられ、100件のサンプル内にすべての地域・階層・通話種別が現れるようにしている。後で加重指標を示せるよう、小さな `subscriber_wt` サーベイウェイトも付与する。

In [1]:
データ work.cdr_billing;
    呼出 streaminit(20260131);
    長さ region $12 plan_tier $24 call_type $12 device_class $24 bill_month $7;
    見出 region        = "地域"
          plan_tier      = "料金プラン"
          call_type      = "通話種別"
          device_class   = "端末区分"
          bill_month     = "請求月"
          revenue        = "決済収益(米ドル)"
          call_minutes   = "課金対象通話分数"
          data_mb        = "データ量(MB)"
          subscriber_wt  = "加入者調査ウェイト";

    繰返 i = 1 から 100;
        /* --- 次元: カテゴリごとに1つの確率、合計1.0 --- */
        r = rand("table", 0.40, 0.33, 0.27);
        選択 (r);
            場合 (1) region = "東部";
            場合 (2) region = "南部";
            その他 region = "北部";
        終了;

        p = rand("table", 0.30, 0.40, 0.30);
        選択 (p);
            場合 (1) plan_tier = "プリペイド";
            場合 (2) plan_tier = "ベーシック";
            その他 plan_tier = "プラス";
        終了;

        c = rand("table", 0.50, 0.30, 0.20);
        選択 (c);
            場合 (1) call_type = "音声";
            場合 (2) call_type = "SMS";
            その他 call_type = "データ";
        終了;

        もし rand("uniform") < 0.55 なら device_class = "スマート";
        他 device_class = "フィーチャー";

        bill_month = "2026-01";

        /* --- 指標、階層とサービスに応じてスケール --- */
        tier_mult = (plan_tier = "プリペイド")*0.7
                  + (plan_tier = "ベーシック")*1.0
                  + (plan_tier = "プラス")*1.7;

        call_minutes = round((call_type = "音声")
                       * rand("gamma", 2.0) * 18 * tier_mult, 0.1);
        data_mb      = round((call_type = "データ")
                       * rand("gamma", 1.5) * 220 * tier_mult, 0.1);

        base_rev = 0.05*call_minutes + 0.010*data_mb
                 + (call_type = "SMS") * rand("poisson", 30) * 0.03;
        revenue = round(base_rev * (0.85 + 0.30*rand("uniform")), 0.01);

        subscriber_wt = round(0.8 + rand("uniform")*1.6, 0.01);

        出力;
    終了;
    削除 i r p c tier_mult base_rev;
実行;

処理 PRINT データ=work.cdr_billing(obs=8) 見出 noobs;
    表題 "加入者日次請求サンプルレコード";
実行;

                                                    加入者日次請求サンプルレコード                                                     

    地域            料金プラン          通話種別                端末区分        請求月                  課金対象通話分数          データ量(MB)                決済収益(米ドル)                    加入者調査ウェイト
北部      プラス              SMS           スマート                2026-01                           0                 0                     0.67                         1.13
南部      プリペイド            SMS           フィーチャー              2026-01                           0                 0                     0.94                         1.42
南部      プラス              SMS           スマート                2026-01                           0                 0                     0.99                         0.86
南部      プラス              SMS           スマート                2026-01                           0                 0                     1.01                         1.03
南部      プラス              音声            スマート


NOTE: DATA work.cdr_billing


NOTE: Wrote work.cdr_billing (100 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.04 seconds
  cpu   0.04 seconds
NOTE: PROC PRINT data=work.cdr_billing

NOTE: PROC PRINT completed: 8 observations printed, 9 variables


## ステップ2 - PROC SUMMARY NWAY で最も細かい粒度のキューブを構築する

`NWAY` は、すべての `CLASS` 次元の組み合わせのうち最も詳細なもの — ここでは実在する (region x plan_tier x call_type) セルすべて — のみを保持する。各セルについて収益の `SUM`、`MEAN`、`MAX` に加え通話分数とメガバイトの合計を格納するので、アナリストはセルごとの総収益を読み取り、レコード当たり平均を導出し、最大の単発課金を見つけられる。`_FREQ_` は各セルの背後にある加入者・日次件数を記録する。NWAY粒度ではすべての行が同じタイプになるため、ここでは `_TYPE_` をドロップする。

In [2]:
処理 summary データ=work.cdr_billing NWAY;
    分類 region plan_tier call_type;
    変数 revenue call_minutes data_mb;
    出力 out=work.cube_nway(drop=_type_)
           sum(revenue)=rev_total  mean(revenue)=rev_mean  MAX(revenue)=rev_max
           sum(call_minutes)=min_total
           sum(data_mb)=data_total;
実行;

処理 PRINT データ=work.cube_nway(obs=12) 見出 noobs;
    見出 region='地域' plan_tier='料金プラン' call_type='通話種別'
          rev_total='収益合計' rev_mean='平均収益' rev_max='最大収益'
          min_total='通話分数合計' data_total='データ量合計';
    表題 "NWAYキューブセル(地域 x 料金プラン x 通話種別)";
    書式 rev_total rev_mean rev_max min_total data_total comma10.2;
実行;

                                             NWAYキューブセル(地域 x 料金プラン x 通話種別)                                              

    地域            料金プラン          通話種別  _FREQ_          収益合計          平均収益          最大収益              通話分数合計              データ量合計
北部      プラス              SMS                4          3.00          0.75          0.97                0.00                0.00
北部      プラス              音声                 3          8.12          2.71          3.80              149.00                0.00
北部      プリペイド            SMS                2          2.00          1.00          1.18                0.00                0.00
北部      プリペイド            データ                2          2.16          1.08          1.09                0.00              229.50
北部      プリペイド            音声                 5          8.56          1.71          2.15              165.70                0.00
北部      ベーシック            SMS                2          1.95          0.97          1.17                0.00   


NOTE: PROC MEANS
NOTE: Output dataset work.cube_nway has 25 observations and 9 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_nway

NOTE: PROC PRINT completed: 12 observations printed, 9 variables


## ステップ3 - TYPES で名前付きドリルサブキューブを具現化する

OLAPキューブは、アナリストが最も頻繁にたどるロールアップを事前に格納しておく。`TYPES` 文はまさにそれを行う: 各項目が `PROC SUMMARY` に1つのサブキューブを出力するよう指示する。ここでは総合計 `()`、`region` のマージナル、そして2元の `region*plan_tier` と `call_type*plan_tier` のサブキューブ — 収益ダッシュボードが公開するであろうドリルパス — を要求する。SAS は各サブキューブに `_TYPE_` コード(`CLASS` リストに対するビットマスク)でタグ付けするので、単一の出力データセットが階層のすべてのレベルを保持する。

In [3]:
処理 summary データ=work.cdr_billing;
    分類 region plan_tier call_type;
    変数 revenue;
    TYPES () region region*plan_tier call_type*plan_tier;
    出力 out=work.cube_hier
           sum(revenue)=rev_total;
実行;

処理 PRINT データ=work.cube_hier 見出 noobs;
    表題 "サブキューブ集計:総合計、地域、地域x階層、通話種別x階層";
    変数 _type_ region plan_tier call_type _freq_ rev_total;
    見出 _type_='タイプ' _freq_='件数' region='地域' plan_tier='料金プラン'
          call_type='通話種別' rev_total='収益合計';
    書式 rev_total comma10.2;
実行;

                                             サブキューブ集計:総合計、地域、地域x階層、通話種別x階層                                              

      タイプ      地域            料金プラン          通話種別      件数          収益合計
        0                                            100        222.78
        3          プラス              SMS               13         11.75
        3          プラス              データ                3         17.46
        3          プラス              音声                18         59.06
        3          プリペイド            SMS                7          6.82
        3          プリペイド            データ                7         14.50
        3          プリペイド            音声                16         24.77
        3          ベーシック            SMS                8          8.03
        3          ベーシック            データ                8         38.06
        3          ベーシック            音声                20         42.33
        4  北部                                         23         38.40
        4  南部             


NOTE: PROC MEANS
NOTE: Output dataset work.cube_hier has 22 observations and 6 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_hier

NOTE: PROC PRINT completed: 22 observations printed, 6 variables


## ステップ4 - 加重投影、地域ドリル、漏えいトリアージ

収益保証チームが実際にキューブに対して行う3つの読み取り:

- **加重投影。** `region*plan_tier` サマリーに `WEIGHT=subscriber_wt` を付与すると、サンプリングされた生の行ではなく、サンプルが代表する加入者ベース全体にスケールした収益が報告される。
- **ドリル。** NWAYキューブを1つの地域でフィルタリングすると、階層の1つの枝 — ここでは南部 — が階層・サービス別の詳細に展開される。
- **漏えいトリアージ。** セルをレコード当たり平均収益でソートすると、最も収益性の高いセルが浮かび上がる。頻度が薄く収益性の高いセルこそ、監査が誤課金や漏えいの疑いで精査する対象である。

In [4]:
/* サブスクライバーベースに投影された加重収益 */
処理 summary データ=work.cdr_billing NWAY;
    分類 region plan_tier;
    変数 revenue;
    重み subscriber_wt;
    出力 out=work.cube_wt(drop=_type_ _freq_)
           sum(revenue)=rev_weighted  n(revenue)=records;
実行;

処理 PRINT データ=work.cube_wt 見出 noobs;
    見出 region='地域' plan_tier='料金プラン'
          rev_weighted='加重収益' records='レコード数';
    表題 "地域x料金プラン別加重収益(加入者ベースに投影)";
    書式 rev_weighted comma10.2;
実行;

/* 南部地域のブランチにドリルダウン */
処理 PRINT データ=work.cube_nway 見出 noobs;
    条件 region = "南部";
    表題 "南部へのドリルダウン:階層・通話種別別の収益セル";
    変数 plan_tier call_type _freq_ rev_total rev_mean;
    見出 plan_tier='料金プラン' call_type='通話種別' _freq_='件数'
          rev_total='収益合計' rev_mean='平均収益';
    書式 rev_total rev_mean comma10.2;
実行;

/* レコード当たり収益でセルをランク付け(漏えいトリアージ) */
処理 SORT データ=work.cube_nway out=work.cube_ranked;
    基準 DESCENDING rev_mean;
実行;

処理 PRINT データ=work.cube_ranked(obs=6) 見出 noobs;
    表題 "平均収益が最も高いセル(レコード当たり収益)";
    変数 region plan_tier call_type _freq_ rev_mean rev_max;
    見出 region='地域' plan_tier='料金プラン' call_type='通話種別'
          _freq_='件数' rev_mean='平均収益' rev_max='最大収益';
    書式 rev_mean rev_max comma10.2;
実行;

                                                地域x料金プラン別加重収益(加入者ベースに投影)                                                

    地域            料金プラン          加重収益            レコード数
北部      プラス                     22.42                7
北部      プリペイド                   20.56                9
北部      ベーシック                   18.30                7
南部      プラス                     56.29               15
南部      プリペイド                   27.77               10
南部      ベーシック                   58.63               14
東部      プラス                     59.59               12
東部      プリペイド                   29.77               11
東部      ベーシック                   50.85               15

                                                南部へのドリルダウン:階層・通話種別別の収益セル                                                

          料金プラン          通話種別      件数          収益合計          平均収益
プラス              SMS                5          5.16          1.03
プラス              データ                2         11.92          5.96
プラス    


NOTE: PROC MEANS
NOTE: Output dataset work.cube_wt has 9 observations and 4 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_wt

NOTE: PROC PRINT completed: 9 observations printed, 4 variables
NOTE: PROC PRINT data=work.cube_nway

NOTE: PROC PRINT completed: 9 observations printed, 5 variables
NOTE: PROC SORT data=work.cube_nway

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 25 rows from work.cube_nway.
NOTE: Wrote work.cube_ranked (25 rows, 9 columns).
NOTE: PROC SORT statement used.
NOTE: PROC PRINT data=work.cube_ranked

NOTE: PROC PRINT completed: 6 observations printed, 6 variables


## 結果の解釈

サマリーキューブは、100件の生の加入者・日次データ行を、ファクトテーブルを再スキャンすることなく即座にドリルダウンの質問に答えられる、コンパクトな事前集計セルの集合に変換する:

- **収益がどこにあるか。** `region` のマージナル(`_TYPE_=4`)は、南部が\$97.44、東部が\$86.94を決済したことを示しており — 合わせて\$222.78の月間収益の83%を占め — 一方、北部は\$38.40であった。`call_type*plan_tier` サブキューブ(`_TYPE_=3`)内では、プラス階層の音声が18件で\$59.06と単一で最も収益性の高いセルであり、次点はベーシック階層の音声で\$42.33であった。
- **加重投影。** サーベイウェイトを適用すると、加入者のウェイトが大きいプランへとランキングが再編される。東部・プラス(\$59.59)と南部・ベーシック(\$58.63)が投影後の `region*plan_tier` 収益で上位となり、加重前の地域合計とは異なる姿を示す — エクスポージャーの規模を測る際にはサンプル収益ではなく投影収益を報告すべきだという教訓である。
- **レコード当たり収益と漏えいトリアージ。** NWAYセルを `rev_mean` でランク付けすると、データ利用セルが上位に来る — 北部・ベーシック・データ(\$7.87/件)と南部・プラス・データ(\$5.96/件)であり、データ利用の多いセルがレコード当たり最高の収益を生むことを裏付けている。頻度の薄い上位セル(1件か2件)こそ、収益保証アナリストが正しく課金されているか、それとも誤りかを確認するために生の通話明細レコードを取り出すべき対象そのものである。

収益保証チームにとって、このキューブは分散ダッシュボードの基盤となる: 決済収益を(地域、階層、通話種別)セルごとの想定レートカード収益と比較し、平均値または加重合計の乖離が最も大きいセルが監査に値する漏えい候補となる。この構造全体が通常の SAS データセットであるため、翌月のキューブも同じ Base SAS ツールで和集合・差分・レートカードとの結合ができる。